# Data-Quality Evidence | Issue #18

This notebook reproduces the Gate 1 source profile from the frozen Allstate source and configuration, verifies the required data-quality results, and generates the Gate 3 evidence artifacts.

- **Authoritative source:** `data/allstate_claims_data.csv`
- **Frozen Gate 1 profile:** `notebooks/final-deliverables/September/Gate 1/profile.csv`
- **Evidence output:** `notebooks/final-deliverables/September/Gate 3/data_quality_evidence/`

The workflow must stop before publishing evidence if source identity, schema, integrity, profile reproduction, or anomaly checks fail.


In [2]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import uuid

import numpy as np
import pandas as pd


## Frozen Source and Profile Configuration

The next cell records the approved Gate 1 source identity, ordered schema, field roles, and validation rules. Keeping these expectations separate from the observed data prevents the workflow from treating an unexpected source as valid merely because it can be loaded.


In [3]:
# find_repo_root() is a utility function that searches for the root of the repository by looking for specific files that should exist in the repo. It starts from the current working directory and moves up the directory tree until it finds a directory containing both the required source CSV file and the required profile CSV file. If it finds such a directory, it returns that path as the repository root. If it does not find such a directory, it raises a FileNotFoundError.
def find_repo_root() -> Path:
    current_path = Path.cwd().resolve()
    required_source = Path("data/allstate_claims_data.csv")
    required_profile = Path(
        "notebooks/final-deliverables/September/Gate 1/profile.csv"
    )

    # Search for the repository root by checking the current directory and its parents
    for candidate in [current_path, *current_path.parents]: # creates a list of current path and parent directories 
        if (candidate / required_source).is_file() and (
            candidate / required_profile
        ).is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the repository containing the source CSV "
        "and frozen Gate 1 profile."
    )


REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data" / "allstate_claims_data.csv"
GATE1_PROFILE_PATH = (
    REPO_ROOT
    / "notebooks"
    / "final-deliverables"
    / "September"
    / "Gate 1"
    / "profile.csv"
)
EVIDENCE_DIR = (
    REPO_ROOT
    / "notebooks"
    / "final-deliverables"
    / "September"
    / "Gate 3"
    / "data_quality_evidence"
)

EXPECTED_SOURCE_BYTES = 70_025_339
EXPECTED_SOURCE_SHA256 = (
    "74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03"
)
EXPECTED_ROWS = 188_318
EXPECTED_COLUMNS = 132

ID_COLUMN = "id"
CATEGORICAL_COLUMNS = [f"cat{i}" for i in range(1, 117)]
CONTINUOUS_COLUMNS = [f"cont{i}" for i in range(1, 15)]
TARGET_COLUMN = "loss"
EXPECTED_HEADER = (
    [ID_COLUMN]
    + CATEGORICAL_COLUMNS
    + CONTINUOUS_COLUMNS
    + [TARGET_COLUMN]
)

EXPECTED_ROLE_COUNTS = {
    "identifier": 1,
    "categorical_predictor": 116,
    "continuous_predictor": 14,
    "regression_target": 1,
}
CONTINUOUS_MINIMUM = 0.0
CONTINUOUS_MAXIMUM = 1.0
CATEGORICAL_ORDERED = False

PROFILE_COLUMNS = [
    "column",
    "role",
    "source_dtype",
    "workflow_dtype",
    "missing",
    "n_unique",
    "min",
    "max",
]
ANOMALY_COLUMNS = [
    "id",
    "description",
    "affected_count",
    "severity",
    "owner",
    "status",
    "evidence",
    "handling_decision",
    "likely_impact",
]


## Source Identity Verification

Verify the file size and SHA-256 digest before reading the CSV. A matching shape or header alone is not enough to establish that this is the frozen source approved for Gate 1. The workflow must stop immediately if either identity check fails.


In [4]:
# calculates the SHA-256 hash of a file at the given path. The function returns the hash, reads the file in chunks for efficiency.
def calculate_sha256(path, chunk_size=1024 * 1024) -> str:
    digest = hashlib.sha256() 

    with path.open("rb") as source_file:
        for chunk in iter(lambda: source_file.read(chunk_size), b""):
            digest.update(chunk)

    return digest.hexdigest() # returns the hexadecimal representation of the hash


observed_source_bytes = DATA_PATH.stat().st_size
observed_source_sha256 = calculate_sha256(DATA_PATH)

source_identity_results = pd.DataFrame(
    [
        {
            "check": "source file size",
            "observed": str(observed_source_bytes),
            "expected": str(EXPECTED_SOURCE_BYTES),
            "status": (
                "pass"
                if observed_source_bytes == EXPECTED_SOURCE_BYTES
                else "fail"
            ),
        },
        {
            "check": "source SHA-256",
            "observed": observed_source_sha256,
            "expected": EXPECTED_SOURCE_SHA256,
            "status": (
                "pass"
                if observed_source_sha256 == EXPECTED_SOURCE_SHA256
                else "fail"
            ),
        },
    ]
)

display(source_identity_results)

if not source_identity_results["status"].eq("pass").all():
    raise RuntimeError(
        "Source identity verification failed. Stop the workflow and "
        "record the mismatch as a blocking anomaly."
    )

print("SOURCE IDENTITY: PASS")


,check,observed,expected,status
0,source file size,70025339,70025339,pass
1,source SHA-256,74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf...,74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf...,pass


SOURCE IDENTITY: PASS


## Load the Source and Verify Its Schema

Load the identity-verified CSV without modifying source values. Before profiling, confirm the exact row count, column count, and ordered header. A schema mismatch is blocking because every later role, integrity, and range check depends on the approved 132-column structure.


In [5]:
raw_df = pd.read_csv(
    DATA_PATH, 
    dtype={column: "object" for column in CATEGORICAL_COLUMNS} # remember to read categorical columns as object type 
)

# confirm that the number of rows, columns, and the header of the source data matches the expected values. If any of these checks fail, a RuntimeError is raised to stop the workflow and record the mismatch as a blocking anomaly.
observed_rows, observed_columns = raw_df.shape
observed_header = raw_df.columns.tolist()
header_matches = observed_header == EXPECTED_HEADER

schema_results = pd.DataFrame(
    [
        {
            "check": "row count",
            "observed": observed_rows,
            "expected": EXPECTED_ROWS,
            "status": "pass" if observed_rows == EXPECTED_ROWS else "fail",
        },
        {
            "check": "column count",
            "observed": observed_columns,
            "expected": EXPECTED_COLUMNS,
            "status": (
                "pass" if observed_columns == EXPECTED_COLUMNS else "fail"
            ),
        },
        {
            "check": "ordered header",
            "observed": header_matches,
            "expected": True,
            "status": "pass" if header_matches else "fail",
        },
    ]
)

display(schema_results)

if not schema_results["status"].eq("pass").all():
    raise RuntimeError(
        "Source schema verification failed. Stop the workflow and "
        "record the mismatch as a blocking anomaly."
    )

print(f"SOURCE SCHEMA: PASS — {observed_rows:,} rows × {observed_columns} columns")


,check,observed,expected,status
0,row count,188318,188318,pass
1,column count,132,132,pass
2,ordered header,True,True,pass


SOURCE SCHEMA: PASS — 188,318 rows × 132 columns


## Verify Feature Roles

Assign roles from the frozen configuration, not from inferred data types: `id` is the identifier, `cat1`–`cat116` are categorical predictors, `cont1`–`cont14` are continuous predictors, and `loss` is the regression target. Verify both complete one-to-one column coverage and the expected role counts before applying workflow dtypes. Any missing, extra, or multiply assigned column is a blocking anomaly.


In [6]:
# verify that each column in the source data has been assigned a role, that there are no unexpected columns, and that no column has been assigned multiple roles. The results of these checks are displayed in a DataFrame, and if any of the checks fail, a RuntimeError is raised to stop the workflow and record the mismatch as a blocking anomaly.

# this generates a list of tuples where each tuple contains a column name and its assigned role.
role_assignments = (
    [(ID_COLUMN, "identifier")]
    + [(column, "categorical_predictor") for column in CATEGORICAL_COLUMNS]
    + [(column, "continuous_predictor") for column in CONTINUOUS_COLUMNS]
    + [(TARGET_COLUMN, "regression_target")]
)

assigned_columns = [column for column, _ in role_assignments] 
assignment_counts = pd.Series(assigned_columns).value_counts() 

missing_role_columns = sorted(set(raw_df.columns) - set(assigned_columns))
unexpected_role_columns = sorted(set(assigned_columns) - set(raw_df.columns))
multiply_assigned_columns = sorted(
    assignment_counts[assignment_counts > 1].index.tolist()
)
role_by_column = dict(role_assignments)
observed_role_counts = pd.Series(role_by_column).value_counts().to_dict()

role_results = pd.DataFrame(
    [
        {
            "role": role,
            "observed_count": observed_role_counts.get(role, 0),
            "expected_count": expected_count,
            "status": (
                "pass"
                if observed_role_counts.get(role, 0) == expected_count
                else "fail"
            ),
        }
        for role, expected_count in EXPECTED_ROLE_COUNTS.items()
    ]
) 
coverage_passes = not ( 
    missing_role_columns
    or unexpected_role_columns
    or multiply_assigned_columns
)

display(role_results)
print(f"One-to-one column coverage: {'PASS' if coverage_passes else 'FAIL'}")

if not coverage_passes or not role_results["status"].eq("pass").all():
    raise RuntimeError(
        "Feature-role verification failed. "
        f"Missing assignments: {missing_role_columns}; "
        f"unknown configured columns: {unexpected_role_columns}; "
        f"multiple assignments: {multiply_assigned_columns}."
    )

print("FEATURE ROLES: PASS")


,role,observed_count,expected_count,status
0,identifier,1,1,pass
1,categorical_predictor,116,116,pass
2,continuous_predictor,14,14,pass
3,regression_target,1,1,pass


One-to-one column coverage: PASS
FEATURE ROLES: PASS


## Reproduce the Frozen Gate 1 Profile

Create a working copy and convert only the 116 configured categorical predictors to pandas `category` dtype. Preserve `raw_df` unchanged so the profile can report both the source dtype and the configured workflow dtype. Then calculate, in source-column order, each column's role, missing count, distinct-value count, and observed numeric minimum and maximum.


In [7]:
workflow_df = raw_df.copy() # create a copy of the raw DataFrame to apply transformations and maintain the original data intact.

for column in CATEGORICAL_COLUMNS:
    workflow_df[column] = pd.Categorical(
        workflow_df[column].astype("string"), # convert the column to string type before converting it to categorical type
        ordered=CATEGORICAL_ORDERED,
    )

profile_rows = []

for column in raw_df.columns:
    source_series = raw_df[column]
    workflow_series = workflow_df[column]
    is_numeric = pd.api.types.is_numeric_dtype(source_series)

    profile_rows.append(
        {
            "column": column,
            "role": role_by_column[column],
            "source_dtype": str(source_series.dtype),
            "workflow_dtype": str(workflow_series.dtype),
            "missing": int(source_series.isna().sum()),
            "n_unique": int(source_series.nunique(dropna=False)),
            "min": source_series.min() if is_numeric else None,
            "max": source_series.max() if is_numeric else None,
        }
    )

reproduced_profile = pd.DataFrame(profile_rows, columns=PROFILE_COLUMNS)

print(f"Converted categorical fields: {len(CATEGORICAL_COLUMNS)}")
print(f"Profile rows: {len(reproduced_profile)}")
display(reproduced_profile)


Converted categorical fields: 116
Profile rows: 132


,column,role,source_dtype,workflow_dtype,missing,n_unique,min,max
0,id,identifier,int64,int64,0,188318,1.000000,587633.000000
1,cat1,categorical_predictor,object,category,0,2,NaN,NaN
2,cat2,categorical_predictor,object,category,0,2,NaN,NaN
3,cat3,categorical_predictor,object,category,0,2,NaN,NaN
4,cat4,categorical_predictor,object,category,0,2,NaN,NaN
...,...,...,...,...,...,...,...,...
127,cont11,continuous_predictor,float64,float64,0,326,0.035321,0.998742
128,cont12,continuous_predictor,float64,float64,0,328,0.036232,0.998484
129,cont13,continuous_predictor,float64,float64,0,353,0.000228,0.988494
130,cont14,continuous_predictor,float64,float64,0,18740,0.179722,0.844848


## Confirm Exact Profile Reproduction

Load the frozen Gate 1 profile and compare it with the newly reproduced profile in the same column and row order. The comparison includes roles, source and workflow dtypes, missingness, distinct counts, and numeric ranges. Any difference is a blocking reproducibility anomaly and must stop evidence publication.


In [8]:
frozen_profile = pd.read_csv(GATE1_PROFILE_PATH)
profile_comparison_detail = ""

# this block compares the reproduced profile with the frozen profile. It uses pandas' testing utility to assert that the two DataFrames are equal, ignoring the index and checking for exact matches in data types and values. If they match, it sets `exact_profile_match` to True; if not, it catches the AssertionError, sets `exact_profile_match` to False, and records the error message for later reporting.
try:
    pd.testing.assert_frame_equal(
        reproduced_profile.reset_index(drop=True),
        frozen_profile.reset_index(drop=True),
        check_dtype=True,
        check_exact=True,
    )
    exact_profile_match = True
except AssertionError as error:
    exact_profile_match = False
    profile_comparison_detail = str(error)

profile_reproduction_results = pd.DataFrame(
    [
        {
            "check": "profile row count",
            "observed": len(reproduced_profile),
            "expected": len(frozen_profile),
            "status": (
                "pass"
                if len(reproduced_profile) == len(frozen_profile)
                else "fail"
            ),
        },
        {
            "check": "ordered profile columns",
            "observed": reproduced_profile.columns.tolist() == PROFILE_COLUMNS,
            "expected": True,
            "status": (
                "pass"
                if reproduced_profile.columns.tolist() == PROFILE_COLUMNS
                else "fail"
            ),
        },
        {
            "check": "exact frozen profile match",
            "observed": exact_profile_match,
            "expected": True,
            "status": "pass" if exact_profile_match else "fail",
        },
    ]
)

display(profile_reproduction_results)

if not profile_reproduction_results["status"].eq("pass").all():
    if profile_comparison_detail:
        print(profile_comparison_detail)
    raise RuntimeError(
        "Frozen Gate 1 profile reproduction failed. Stop the workflow "
        "and record the difference as a blocking anomaly."
    )

print("FROZEN PROFILE REPRODUCTION: PASS")


,check,observed,expected,status
0,profile row count,132,132,pass
1,ordered profile columns,True,True,pass
2,exact frozen profile match,True,True,pass


FROZEN PROFILE REPRODUCTION: PASS


### Environment Compatibility Note

The current pandas version initially inferred CSV text columns as `str`, whereas the frozen Gate 1 environment inferred them as `object`. This affected the recorded `source_dtype` label for 116 categorical columns but did not affect values, roles, missingness, uniqueness, or numeric ranges. The categorical source columns are therefore loaded explicitly as `object` to reproduce the frozen configuration deterministically. This is an environment-compatibility difference, not a data anomaly, and is not entered in the anomaly register.


## Verify Dataset Integrity

Confirm that `id` is present, non-missing, and unique across all 188,318 rows. Also calculate total cell-level missingness and the number of fully duplicated rows. Any missing identifier, duplicate identifier, missing value, or duplicate row is blocking because it would contradict the frozen clean-data baseline.


In [9]:
identifier_row_count = int(raw_df[ID_COLUMN].size)
identifier_missing_count = int(raw_df[ID_COLUMN].isna().sum())
identifier_unique_count = int(raw_df[ID_COLUMN].nunique(dropna=False))
identifier_duplicate_count = int(raw_df[ID_COLUMN].duplicated().sum())
total_missing_cells = int(raw_df.isna().sum().sum())
duplicate_row_count = int(raw_df.duplicated().sum())

integrity_results = pd.DataFrame(
    [
        {
            "check": "identifier row count",
            "observed": identifier_row_count,
            "expected": EXPECTED_ROWS,
        },
        {
            "check": "identifier missing count",
            "observed": identifier_missing_count,
            "expected": 0,
        },
        {
            "check": "identifier unique count",
            "observed": identifier_unique_count,
            "expected": EXPECTED_ROWS,
        },
        {
            "check": "repeated identifier count",
            "observed": identifier_duplicate_count,
            "expected": 0,
        },
        {
            "check": "total missing cells",
            "observed": total_missing_cells,
            "expected": 0,
        },
        {
            "check": "fully duplicated rows",
            "observed": duplicate_row_count,
            "expected": 0,
        },
    ]
)
integrity_results["status"] = np.where(
    integrity_results["observed"] == integrity_results["expected"],
    "pass",
    "fail",
)

display(integrity_results)
print(f"Identifier is unique: {raw_df[ID_COLUMN].is_unique}")

if not integrity_results["status"].eq("pass").all():
    raise RuntimeError(
        "Dataset integrity verification failed. Stop the workflow and "
        "record each failed check as a blocking anomaly."
    )

print("DATASET INTEGRITY: PASS")


,check,observed,expected,status
0,identifier row count,188318,188318,pass
1,identifier missing count,0,0,pass
2,identifier unique count,188318,188318,pass
3,repeated identifier count,0,0,pass
4,total missing cells,0,0,pass
5,fully duplicated rows,0,0,pass


Identifier is unique: True
DATASET INTEGRITY: PASS


## Verify Observed Numeric Ranges

Report the observed minimum and maximum for all 16 numeric fields: the identifier, 14 continuous predictors, and regression target. Compare every observed endpoint with the frozen Gate 1 profile and separately confirm that each continuous predictor remains within the configured `[0, 1]` domain. A changed endpoint or out-of-domain continuous value is blocking.


In [10]:
numeric_columns = [ID_COLUMN] + CONTINUOUS_COLUMNS + [TARGET_COLUMN]
frozen_ranges = frozen_profile.set_index("column")[["min", "max"]] # ranges are frozen in the profile for each numeric column 
numeric_range_rows = []

for column in numeric_columns: 
    # for each column, calculate 

    observed_minimum = raw_df[column].min()
    observed_maximum = raw_df[column].max()
    frozen_minimum = frozen_ranges.at[column, "min"]
    frozen_maximum = frozen_ranges.at[column, "max"]
    endpoints_match = (
        observed_minimum == frozen_minimum
        and observed_maximum == frozen_maximum
    )

    if column in CONTINUOUS_COLUMNS:
        domain_rule = "[0, 1]"
        domain_passes = (
            observed_minimum >= CONTINUOUS_MINIMUM
            and observed_maximum <= CONTINUOUS_MAXIMUM
        )
    else:
        domain_rule = "observed only"
        domain_passes = True

    numeric_range_rows.append(
        {
            "column": column,
            "role": role_by_column[column],
            "observed_min": observed_minimum,
            "observed_max": observed_maximum,
            "frozen_min": frozen_minimum,
            "frozen_max": frozen_maximum,
            "domain_rule": domain_rule,
            "status": (
                "pass" if endpoints_match and domain_passes else "fail"
            ),
        }
    )

numeric_range_results = pd.DataFrame(numeric_range_rows)
display(numeric_range_results)

if len(numeric_range_results) != 16 or not numeric_range_results["status"].eq("pass").all():
    raise RuntimeError(
        "Numeric-range verification failed. Stop the workflow and "
        "record each failed field as a blocking anomaly."
    )

print("NUMERIC RANGES: PASS — 16 fields verified")


,column,role,observed_min,observed_max,frozen_min,frozen_max,domain_rule,status
0,id,identifier,1.000000,587633.000000,1.000000,587633.000000,observed only,pass
1,cont1,continuous_predictor,0.000016,0.984975,0.000016,0.984975,"[0, 1]",pass
2,cont2,continuous_predictor,0.001149,0.862654,0.001149,0.862654,"[0, 1]",pass
3,cont3,continuous_predictor,0.002634,0.944251,0.002634,0.944251,"[0, 1]",pass
4,cont4,continuous_predictor,0.176921,0.954297,0.176921,0.954297,"[0, 1]",pass
5,cont5,continuous_predictor,0.281143,0.983674,0.281143,0.983674,"[0, 1]",pass
6,cont6,continuous_predictor,0.012683,0.997162,0.012683,0.997162,"[0, 1]",pass
7,cont7,continuous_predictor,0.069503,1.000000,0.069503,1.000000,"[0, 1]",pass
8,cont8,continuous_predictor,0.236880,0.980200,0.236880,0.980200,"[0, 1]",pass
9,cont9,continuous_predictor,0.000080,0.995400,0.000080,0.995400,"[0, 1]",pass


NUMERIC RANGES: PASS — 16 fields verified


## Clean Data Is Not Analysis-Complete

The verified absence of missing values, duplicate rows, identifier defects, schema drift, and range violations substantially reduces data-repair work. It does **not** establish that the predictors are statistically well supported or useful for modeling. EDA must still examine categorical level support, continuous and target distributions, and predictor–target relationships. Sparse levels, skew, heavy tails, outliers, nonlinear patterns, and weak or unstable relationships can remain in a structurally clean dataset and may require modeling rules rather than source-data repair.


## Examine Categorical Level Support

Reuse the Gate 2 support rule: flag a categorical level as rare when it has fewer than **188 claims**, approximately 0.1% of the 188,318 rows. Also flag fields where one level represents more than **99%** of rows. Report field-level cardinality, minimum support, rare-level count, affected-row count and share, and dominant-level share. These are non-blocking modeling risks: preserve all rows and levels, and use the findings to guide rare-level pooling, encoding, validation, and interpretation rather than source-data repair.


In [ ]:
# Note: A categorical level is defined as rare when it appears in fewer than 188 claims, which is approximately 0.1% of the 188,318 rows

RARE_LEVEL_COUNT_THRESHOLD = 188
DOMINANT_LEVEL_SHARE_THRESHOLD = 0.99

categorical_support_rows = []
rows_with_any_rare_level = pd.Series(False, index=raw_df.index)

for column in CATEGORICAL_COLUMNS:
    counts = raw_df[column].value_counts(dropna=False)
    rare_counts = counts[counts < RARE_LEVEL_COUNT_THRESHOLD]
    rare_values = rare_counts.index
    rare_row_count = int(rare_counts.sum())
    rows_with_any_rare_level |= raw_df[column].isin(rare_values)

    categorical_support_rows.append(
        {
            "column": column,
            "level_count": int(counts.size),
            "minimum_level_support": int(counts.min()),
            "rare_level_count": int(rare_counts.size),
            "rare_row_count": rare_row_count,
            "rare_row_share": rare_row_count / len(raw_df),
            "dominant_level": counts.index[0],
            "dominant_level_count": int(counts.iloc[0]),
            "dominant_level_share": float(counts.iloc[0] / len(raw_df)),
            "has_rare_level": bool(rare_counts.size),
            "has_dominant_level": bool(
                counts.iloc[0] / len(raw_df)
                > DOMINANT_LEVEL_SHARE_THRESHOLD
            ),
        }
    )

categorical_support_results = pd.DataFrame(categorical_support_rows)
flagged_categorical_support = categorical_support_results.loc[
    categorical_support_results["has_rare_level"]
    | categorical_support_results["has_dominant_level"]
].copy()

categorical_support_summary = pd.DataFrame(
    [
        {"metric": "categorical fields assessed", "count": len(categorical_support_results)},
        {
            "metric": "fields with at least one rare level",
            "count": int(categorical_support_results["has_rare_level"].sum()),
        },
        {
            "metric": "rare levels",
            "count": int(categorical_support_results["rare_level_count"].sum()),
        },
        {
            "metric": "distinct rows with at least one rare level",
            "count": int(rows_with_any_rare_level.sum()),
        },
        {
            "metric": "fields with a level above 99% share",
            "count": int(categorical_support_results["has_dominant_level"].sum()),
        },
    ]
)

display(categorical_support_summary)
display(
    flagged_categorical_support.sort_values(
        ["has_rare_level", "rare_level_count", "dominant_level_share"],
        ascending=[False, False, False],
    )
)

if len(categorical_support_results) != len(CATEGORICAL_COLUMNS):
    raise RuntimeError("Categorical support inventory is incomplete.")

print("CATEGORICAL LEVEL SUPPORT: COMPLETE")


,metric,count
0,categorical fields assessed,116
1,fields with at least one rare level,39
2,rare levels,523
3,distinct rows with at least one rare level,14721
4,fields with a level above 99% share,31


,column,level_count,minimum_level_support,rare_level_count,rare_row_count,rare_row_share,dominant_level,dominant_level_count,dominant_level_share,has_rare_level,has_dominant_level
115,cat116,326,1,223,6673,0.035435,HK,21061,0.111837,True,False
109,cat110,131,1,92,2916,0.015484,CL,25305,0.134374,True,False
108,cat109,84,1,71,2791,0.014821,BI,152918,0.812020,True,False
112,cat113,61,1,20,387,0.002055,BM,26191,0.139079,True,False
104,cat105,20,1,11,494,0.002623,E,76493,0.406191,True,False
114,cat115,23,1,9,186,0.000988,K,43866,0.232936,True,False
113,cat114,19,1,8,94,0.000499,A,131693,0.699312,True,False
110,cat111,16,2,8,176,0.000935,A,128395,0.681799,True,False
100,cat101,19,1,8,326,0.001731,A,106721,0.566706,True,False
106,cat107,20,2,8,481,0.002554,F,47310,0.251224,True,False


CATEGORICAL LEVEL SUPPORT: COMPLETE


## Examine Continuous and Target Distributions

Summarize the 14 continuous predictors and `loss` with count, mean, standard deviation, quartiles, IQR, endpoints, unique-value count, and skewness. For triage, flag absolute skewness above **1** as strong skew and count observations outside the conventional **1.5 × IQR** fences. These flags describe distribution shape and potential modeling sensitivity; they do not prove erroneous values and do not authorize deletion or clipping. A zero-IQR or non-finite summary is treated as a blocking analytical failure because downstream binning and relationship summaries may then be invalid.


In [12]:
DISTRIBUTION_FIELDS = CONTINUOUS_COLUMNS + [TARGET_COLUMN]
STRONG_SKEW_THRESHOLD = 1.0
IQR_FENCE_MULTIPLIER = 1.5
distribution_rows = []

for column in DISTRIBUTION_FIELDS:
    series = raw_df[column]
    q1, median, q3 = series.quantile([0.25, 0.50, 0.75]).tolist()
    iqr = q3 - q1
    lower_fence = q1 - IQR_FENCE_MULTIPLIER * iqr
    upper_fence = q3 + IQR_FENCE_MULTIPLIER * iqr
    outlier_mask = (series < lower_fence) | (series > upper_fence)
    skewness = float(series.skew())

    distribution_rows.append(
        {
            "column": column,
            "role": role_by_column[column],
            "count": int(series.count()),
            "unique_count": int(series.nunique(dropna=False)),
            "mean": float(series.mean()),
            "std": float(series.std()),
            "min": float(series.min()),
            "q1": float(q1),
            "median": float(median),
            "q3": float(q3),
            "max": float(series.max()),
            "iqr": float(iqr),
            "skewness": skewness,
            "strong_skew": abs(skewness) > STRONG_SKEW_THRESHOLD,
            "lower_iqr_fence": float(lower_fence),
            "upper_iqr_fence": float(upper_fence),
            "iqr_outlier_count": int(outlier_mask.sum()),
            "iqr_outlier_share": float(outlier_mask.mean()),
        }
    )

distribution_results = pd.DataFrame(distribution_rows)
finite_summary_columns = [
    "mean", "std", "min", "q1", "median", "q3",
    "max", "iqr", "skewness",
]
distribution_results["summary_finite"] = np.isfinite(
    distribution_results[finite_summary_columns]
).all(axis=1)
distribution_results["positive_iqr"] = distribution_results["iqr"] > 0

display(distribution_results)
print(
    "Strongly skewed fields: "
    f"{int(distribution_results['strong_skew'].sum())} of "
    f"{len(distribution_results)}"
)
print(
    "Fields with IQR-flagged observations: "
    f"{int(distribution_results['iqr_outlier_count'].gt(0).sum())} of "
    f"{len(distribution_results)}"
)

blocking_distribution_failure = (
    len(distribution_results) != len(DISTRIBUTION_FIELDS)
    or not distribution_results["summary_finite"].all()
    or not distribution_results["positive_iqr"].all()
)

if blocking_distribution_failure:
    raise RuntimeError(
        "Distribution analysis is incomplete or contains a non-finite "
        "summary or zero-IQR field. Resolve before continuing."
    )

print("DISTRIBUTION ANALYSIS: COMPLETE")


,column,role,count,unique_count,mean,std,min,q1,median,q3,max,iqr,skewness,strong_skew,lower_iqr_fence,upper_iqr_fence,iqr_outlier_count,iqr_outlier_share,summary_finite,positive_iqr
0,cont1,continuous_predictor,188318,647,0.493861,0.187640,0.000016,0.346090,0.475784,0.623912,0.984975,0.277822,0.516424,False,-0.070643,1.040645,0,0.000000,True,True
1,cont2,continuous_predictor,188318,33,0.507188,0.207202,0.001149,0.358319,0.555782,0.681761,0.862654,0.323442,-0.310941,False,-0.126844,1.166924,0,0.000000,True,True
2,cont3,continuous_predictor,188318,76,0.498918,0.202105,0.002634,0.336963,0.527991,0.634224,0.944251,0.297261,-0.010002,False,-0.108928,1.080116,0,0.000000,True,True
3,cont4,continuous_predictor,188318,112,0.491812,0.211292,0.176921,0.327354,0.452887,0.652072,0.954297,0.324718,0.416096,False,-0.159723,1.139149,0,0.000000,True,True
4,cont5,continuous_predictor,188318,141,0.487428,0.209027,0.281143,0.281143,0.422268,0.643315,0.983674,0.362172,0.681622,False,-0.262115,1.186573,0,0.000000,True,True
5,cont6,continuous_predictor,188318,2573,0.490945,0.205273,0.012683,0.336105,0.440945,0.655021,0.997162,0.318916,0.461214,False,-0.142269,1.133395,0,0.000000,True,True
6,cont7,continuous_predictor,188318,5632,0.484970,0.178450,0.069503,0.350175,0.438285,0.591045,1.000000,0.240870,0.826053,False,-0.011130,0.952350,2659,0.014120,True,True
7,cont8,continuous_predictor,188318,201,0.486437,0.199370,0.236880,0.312800,0.441060,0.623580,0.980200,0.310780,0.676634,False,-0.153370,1.089750,0,0.000000,True,True
8,cont9,continuous_predictor,188318,347,0.485506,0.181660,0.000080,0.358970,0.441450,0.566820,0.995400,0.207850,1.072429,True,0.047195,0.878595,13194,0.070062,True,True
9,cont10,continuous_predictor,188318,174,0.498066,0.185877,0.000000,0.364580,0.461190,0.614590,0.994980,0.250010,0.355001,False,-0.010435,0.989605,140,0.000743,True,True


Strongly skewed fields: 2 of 15
Fields with IQR-flagged observations: 4 of 15
DISTRIBUTION ANALYSIS: COMPLETE


## Continuous Predictor–Target Relationships

For each of the 14 continuous predictors, evaluate linear and rank relationships with both raw `loss` and `log1p(loss)`. Pearson correlation measures linear association, while Spearman correlation captures monotonic association and is less sensitive to the target's long right tail.

Also divide each predictor into up to 10 quantile bins and summarize target count, mean, and median within each bin. `duplicates="drop"` is intentional: tied quantile boundaries may yield fewer than 10 bins without losing observations. The actual bin count and assigned-row count must be retained so this behavior is visible and verifiable. A missing correlation, an unassigned row, or fewer than two usable bins is blocking; fewer than 10 bins caused only by duplicate edges is a resolved, non-blocking workflow condition.

In [13]:
REQUESTED_QUANTILE_BINS = 10
target = raw_df[TARGET_COLUMN]
log_target = np.log1p(target)
relationship_rows = []
quantile_bin_frames = []

for column in CONTINUOUS_COLUMNS:
    predictor = raw_df[column]
    bin_codes, bin_edges = pd.qcut(
        predictor,
        q=REQUESTED_QUANTILE_BINS,
        labels=False,
        retbins=True,
        duplicates="drop",
    )
    actual_bin_count = len(bin_edges) - 1
    assigned_row_count = int(bin_codes.notna().sum())
    unassigned_row_count = int(bin_codes.isna().sum())

    relationship_rows.append(
        {
            "predictor": column,
            "pearson_loss": float(predictor.corr(target, method="pearson")),
            "spearman_loss": float(predictor.corr(target, method="spearman")),
            "pearson_log1p_loss": float(
                predictor.corr(log_target, method="pearson")
            ),
            "spearman_log1p_loss": float(
                predictor.corr(log_target, method="spearman")
            ),
            "requested_quantile_bins": REQUESTED_QUANTILE_BINS,
            "actual_quantile_bins": actual_bin_count,
            "assigned_row_count": assigned_row_count,
            "unassigned_row_count": unassigned_row_count,
            "quantile_bins_reduced": (
                actual_bin_count < REQUESTED_QUANTILE_BINS
            ),
        }
    )

    bin_frame = pd.DataFrame(
        {
            "predictor": column,
            "quantile_bin": bin_codes,
            "loss": target,
            "log1p_loss": log_target,
        }
    )
    bin_summary = (
        bin_frame.groupby(
            ["predictor", "quantile_bin"],
            observed=True,
            as_index=False,
        )
        .agg(
            row_count=("loss", "size"),
            mean_loss=("loss", "mean"),
            median_loss=("loss", "median"),
            mean_log1p_loss=("log1p_loss", "mean"),
            median_log1p_loss=("log1p_loss", "median"),
        )
    )
    quantile_bin_frames.append(bin_summary)

continuous_target_relationships = pd.DataFrame(relationship_rows)
continuous_quantile_bin_results = pd.concat(
    quantile_bin_frames, ignore_index=True
)
reduced_quantile_bins = continuous_target_relationships.loc[
    continuous_target_relationships["quantile_bins_reduced"],
    ["predictor", "requested_quantile_bins", "actual_quantile_bins"],
]

display(continuous_target_relationships)
display(reduced_quantile_bins)

correlation_columns = [
    "pearson_loss",
    "spearman_loss",
    "pearson_log1p_loss",
    "spearman_log1p_loss",
]
blocking_relationship_failure = (
    len(continuous_target_relationships) != len(CONTINUOUS_COLUMNS)
    or not np.isfinite(
        continuous_target_relationships[correlation_columns]
    ).all().all()
    or continuous_target_relationships["unassigned_row_count"].ne(0).any()
    or continuous_target_relationships["actual_quantile_bins"].lt(2).any()
)

if blocking_relationship_failure:
    raise RuntimeError(
        "Continuous target-relationship evidence is incomplete. "
        "Resolve missing correlations, unassigned rows, or unusable bins."
    )

print(
    "Predictors with duplicate-edge bin reductions: "
    f"{len(reduced_quantile_bins)}"
)
print("CONTINUOUS TARGET RELATIONSHIPS: COMPLETE")


,predictor,pearson_loss,spearman_loss,pearson_log1p_loss,spearman_log1p_loss,requested_quantile_bins,actual_quantile_bins,assigned_row_count,unassigned_row_count,quantile_bins_reduced
0,cont1,-0.010237,-0.017641,-0.007335,-0.017641,10,10,188318,0,False
1,cont2,0.141528,0.080066,0.104666,0.080066,10,9,188318,0,True
2,cont3,0.111053,0.068353,0.081548,0.068353,10,10,188318,0,False
3,cont4,-0.035831,-0.027878,-0.027523,-0.027878,10,10,188318,0,False
4,cont5,-0.011355,-0.015114,-0.014958,-0.015114,10,8,188318,0,True
5,cont6,0.040967,0.019697,0.031517,0.019697,10,10,188318,0,False
6,cont7,0.119799,0.054928,0.085095,0.054928,10,10,188318,0,False
7,cont8,0.030508,0.027495,0.032042,0.027495,10,10,188318,0,False
8,cont9,0.014456,0.004401,0.017417,0.004401,10,10,188318,0,False
9,cont10,0.020236,0.003042,0.010604,0.003042,10,10,188318,0,False


,predictor,requested_quantile_bins,actual_quantile_bins
1,cont2,10,9
4,cont5,10,8


Predictors with duplicate-edge bin reductions: 2
CONTINUOUS TARGET RELATIONSHIPS: COMPLETE


## Categorical Predictor–Target Relationships

Summarize `loss` and `log1p(loss)` for every observed level of each categorical predictor. Preserve all levels and observations in the evidence, and mark levels with fewer than 188 rows as rare rather than deleting or combining them during EDA.

For field-level interpretation, compare target means and medians across levels that meet the support threshold. Raw-loss means show sensitivity to costly claims, while medians and log-target means provide more robust views of typical differences. Rare-level target summaries remain available but are not used to make strong relationship claims because their estimates may be unstable. Complete row accounting and finite target summaries are blocking requirements.

In [ ]:
categorical_level_frames = []
categorical_relationship_rows = []

for column in CATEGORICAL_COLUMNS:
    analysis_frame = raw_df[[column, TARGET_COLUMN]].copy()
    analysis_frame["log1p_loss"] = np.log1p(
        analysis_frame[TARGET_COLUMN]
    )
    level_summary = (
        analysis_frame.groupby(column, observed=True, dropna=False)
        .agg(
            row_count=(TARGET_COLUMN, "size"),
            mean_loss=(TARGET_COLUMN, "mean"),
            median_loss=(TARGET_COLUMN, "median"),
            mean_log1p_loss=("log1p_loss", "mean"),
            median_log1p_loss=("log1p_loss", "median"),
        )
        .reset_index()
        .rename(columns={column: "level"})
    )
    level_summary.insert(0, "predictor", column)
    level_summary["is_rare"] = (
        level_summary["row_count"] < RARE_LEVEL_COUNT_THRESHOLD
    )
    categorical_level_frames.append(level_summary)

    supported = level_summary.loc[~level_summary["is_rare"]] # supported means levels that are not rare 
    supported_comparison_available = len(supported) >= 2
    categorical_relationship_rows.append(
        {
            "predictor": column,
            "level_count": len(level_summary),
            "supported_level_count": len(supported),
            "rare_level_count": int(level_summary["is_rare"].sum()),
            "rare_row_count": int(
                level_summary.loc[level_summary["is_rare"], "row_count"].sum()
            ),
            "accounted_row_count": int(level_summary["row_count"].sum()),
            "supported_comparison_available": (
                supported_comparison_available
            ),
            "supported_mean_loss_range": (
                float(supported["mean_loss"].max() - supported["mean_loss"].min())
                if supported_comparison_available
                else np.nan
            ),
            "supported_median_loss_range": (
                float(
                    supported["median_loss"].max()
                    - supported["median_loss"].min()
                )
                if supported_comparison_available
                else np.nan
            ),
            "supported_mean_log1p_loss_range": (
                float(
                    supported["mean_log1p_loss"].max()
                    - supported["mean_log1p_loss"].min()
                )
                if supported_comparison_available
                else np.nan
            ),
        }
    )

categorical_level_target_results = pd.concat(
    categorical_level_frames, ignore_index=True
)
categorical_target_relationships = pd.DataFrame(
    categorical_relationship_rows
)

display(
    categorical_target_relationships.sort_values(
        "supported_median_loss_range", ascending=False
    ).reset_index(drop=True)
)

expected_support = categorical_support_results.set_index("column")
observed_support = categorical_target_relationships.set_index("predictor")
level_counts_match = observed_support["level_count"].equals(
    expected_support.loc[CATEGORICAL_COLUMNS, "level_count"]
)
rare_counts_match = observed_support["rare_level_count"].equals(
    expected_support.loc[CATEGORICAL_COLUMNS, "rare_level_count"]
)
target_summary_columns = [
    "mean_loss",
    "median_loss",
    "mean_log1p_loss",
    "median_log1p_loss",
]
blocking_categorical_relationship_failure = (
    len(categorical_target_relationships) != len(CATEGORICAL_COLUMNS)
    or not categorical_target_relationships["accounted_row_count"].eq(
        len(raw_df)
    ).all()
    or not np.isfinite(
        categorical_level_target_results[target_summary_columns]
    ).all().all()
    or not level_counts_match
    or not rare_counts_match
)

if blocking_categorical_relationship_failure:
    raise RuntimeError(
        "Categorical target-relationship evidence is incomplete or "
        "inconsistent with the level-support results."
    )

print(
    "Fields with at least two supported levels: "
    f"{int(categorical_target_relationships['supported_comparison_available'].sum())} "
    f"of {len(CATEGORICAL_COLUMNS)}"
)
print("CATEGORICAL TARGET RELATIONSHIPS: COMPLETE")


,predictor,level_count,supported_level_count,rare_level_count,rare_row_count,accounted_row_count,supported_comparison_available,supported_mean_loss_range,supported_median_loss_range,supported_mean_log1p_loss_range
0,cat57,2,2,0,0,188318,True,7320.098074,7450.410,1.430195
1,cat89,8,3,5,42,188318,True,5304.349285,5593.290,1.087391
2,cat7,2,2,0,0,188318,True,5286.134183,5540.940,1.085430
3,cat101,19,11,8,326,188318,True,4603.330421,4277.255,1.185236
4,cat114,19,11,8,94,188318,True,4314.431374,4171.540,1.216060
...,...,...,...,...,...,...,...,...,...,...
111,cat62,2,1,1,45,188318,False,NaN,NaN,NaN
112,cat63,2,1,1,79,188318,False,NaN,NaN,NaN
113,cat64,2,1,1,47,188318,False,NaN,NaN,NaN
114,cat68,2,1,1,142,188318,False,NaN,NaN,NaN


Fields with at least two supported levels: 107 of 116
CATEGORICAL TARGET RELATIONSHIPS: COMPLETE


## Anomaly Triage and Resolution

A finding enters the anomaly register only when it affects EDA reliability, interpretation, or a downstream workflow decision. Each non-blocking entry must preserve an affected count, severity, owner, handling rule, likely impact, supporting evidence, and resolution status.

The preceding assertions make source-identity failures, profile mismatches, incomplete row accounting, and invalid summaries blocking: execution stops until they are corrected. Findings such as sparse categorical levels, extreme level dominance, a heavy-tailed target, or tied quantile boundaries are non-blocking when their impact is quantified and an explicit handling rule is recorded. The pandas `str`/`object` inference difference remains an environment-compatibility note, as agreed, and is not recorded as a data anomaly.

In [15]:
blocking_resolution_results = pd.DataFrame(
    [
        {"check": "source identity", "passed": source_identity_results["status"].eq("pass").all()},
        {"check": "source schema", "passed": schema_results["status"].eq("pass").all()},
        {"check": "feature roles", "passed": coverage_passes and role_results["status"].eq("pass").all()},
        {"check": "frozen profile reproduction", "passed": exact_profile_match},
        {"check": "identifier and dataset integrity", "passed": integrity_results["status"].eq("pass").all()},
        {"check": "numeric ranges", "passed": numeric_range_results["status"].eq("pass").all()},
        {"check": "distribution summaries", "passed": not blocking_distribution_failure},
        {"check": "continuous target relationships", "passed": not blocking_relationship_failure},
        {"check": "categorical target relationships", "passed": not blocking_categorical_relationship_failure},
    ]
)
blocking_resolution_results["status"] = np.where(
    blocking_resolution_results["passed"], "resolved", "unresolved"
)
display(blocking_resolution_results)

if not blocking_resolution_results["passed"].all():
    unresolved = blocking_resolution_results.loc[
        ~blocking_resolution_results["passed"], "check"
    ].tolist()
    raise RuntimeError(f"Unresolved blocking anomalies: {unresolved}")

anomaly_rows = []

def add_nonblocking_anomaly(
    description, affected_count, severity, owner, status, evidence,
    handling_decision, likely_impact,
):
    anomaly_rows.append(
        {
            "id": f"EDA-NB-{len(anomaly_rows) + 1:03d}",
            "description": description,
            "affected_count": int(affected_count),
            "severity": severity,
            "owner": owner,
            "status": status,
            "evidence": evidence,
            "handling_decision": handling_decision,
            "likely_impact": likely_impact,
        }
    )

rare_field_count = int(categorical_support_results["has_rare_level"].sum())
rare_level_count = int(categorical_support_results["rare_level_count"].sum())
rare_record_count = int(rows_with_any_rare_level.sum())
if rare_level_count:
    add_nonblocking_anomaly(
        description="Records containing at least one rare categorical level",
        affected_count=rare_record_count,
        severity="medium",
        owner="feature engineering / modeling",
        status="non-blocking; handling defined",
        evidence=(
            f"{rare_level_count} levels below {RARE_LEVEL_COUNT_THRESHOLD} rows "
            f"across {rare_field_count} categorical fields; "
            f"{rare_record_count} distinct source rows affected."
        ),
        handling_decision=(
            "Retain source values; fit any rare-level pooling or encoding on "
            "training folds only and verify unseen-level behavior."
        ),
        likely_impact=(
            "Small groups can yield unstable target summaries and poorly "
            "estimated encoded effects."
        ),
    )

dominant_support = categorical_support_results.loc[
    categorical_support_results["has_dominant_level"]
]
if not dominant_support.empty:
    dominant_evidence = "; ".join(
        f"{row.column}={row.dominant_level_share:.6f}"
        for row in dominant_support.itertuples(index=False)
    )
    add_nonblocking_anomaly(
        description="Categorical fields with one level above 99% share",
        affected_count=len(dominant_support),
        severity="low",
        owner="feature engineering / modeling",
        status="non-blocking; handling defined",
        evidence=dominant_evidence,
        handling_decision=(
            "Retain the fields for reproducibility; assess incremental value "
            "during validation and avoid conclusions driven by tiny minorities."
        ),
        likely_impact=(
            "Near-constant predictors may add little signal and make minority-level "
            "target comparisons unstable."
        ),
    )

strong_skew = distribution_results.loc[distribution_results["strong_skew"]]
if not strong_skew.empty:
    skew_evidence = "; ".join(
        f"{row.column}: skewness={row.skewness:.6f}"
        for row in strong_skew.itertuples(index=False)
    )
    add_nonblocking_anomaly(
        description="Numeric fields with absolute skewness above 1",
        affected_count=len(strong_skew),
        severity="medium",
        owner="EDA / modeling",
        status="non-blocking; handling defined",
        evidence=skew_evidence,
        handling_decision=(
            "Preserve raw values; use robust summaries and compare raw-loss "
            "relationships with log1p-loss relationships."
        ),
        likely_impact=(
            "Means, Pearson correlations, and fitted objectives may be "
            "disproportionately influenced by the upper tail."
        ),
    )

iqr_tail_fields = distribution_results.loc[
    distribution_results["iqr_outlier_count"] > 0
]
if not iqr_tail_fields.empty:
    iqr_tail_count = int(iqr_tail_fields["iqr_outlier_count"].sum())
    iqr_evidence = "; ".join(
        f"{row.column}: {row.iqr_outlier_count} ({row.iqr_outlier_share:.6%})"
        for row in iqr_tail_fields.itertuples(index=False)
    )
    add_nonblocking_anomaly(
        description="Row-field observations outside 1.5 x IQR fences",
        affected_count=iqr_tail_count,
        severity="medium",
        owner="EDA / modeling",
        status="non-blocking; handling defined",
        evidence=iqr_evidence,
        handling_decision=(
            "Treat as distribution-tail evidence, not errors; do not delete or "
            "clip without model-validation evidence."
        ),
        likely_impact=(
            "Tail observations may increase sensitivity of averages, correlations, "
            "and fitted models."
        ),
    )

if not reduced_quantile_bins.empty:
    bin_evidence = "; ".join(
        f"{row.predictor}: {row.actual_quantile_bins} of "
        f"{row.requested_quantile_bins} bins"
        for row in reduced_quantile_bins.itertuples(index=False)
    )
    add_nonblocking_anomaly(
        description="Predictors with quantile bins reduced by tied edges",
        affected_count=len(reduced_quantile_bins),
        severity="low",
        owner="EDA workflow",
        status="resolved",
        evidence=bin_evidence,
        handling_decision=(
            "Use qcut with duplicates='drop', retain the actual bin count, and "
            "verify that every source row is assigned."
        ),
        likely_impact=(
            "The affected predictors have fewer comparison groups than requested, "
            "but no observations are lost."
        ),
    )

unsupported_comparisons = categorical_target_relationships.loc[
    ~categorical_target_relationships["supported_comparison_available"]
]
if not unsupported_comparisons.empty:
    comparison_evidence = "; ".join(
        f"{row.predictor}: {row.supported_level_count} supported of "
        f"{row.level_count} observed levels"
        for row in unsupported_comparisons.itertuples(index=False)
    )
    add_nonblocking_anomaly(
        description="Categorical fields lacking two supported levels for comparison",
        affected_count=len(unsupported_comparisons),
        severity="medium",
        owner="EDA / modeling",
        status="non-blocking; interpretation constrained",
        evidence=comparison_evidence,
        handling_decision=(
            "Retain all level summaries but do not make strong target-relationship "
            "claims where fewer than two levels meet the support threshold."
        ),
        likely_impact=(
            "Apparent target differences are driven by sparse groups and are not "
            "reliable for field-level interpretation."
        ),
    )

anomaly_register = pd.DataFrame(anomaly_rows, columns=ANOMALY_COLUMNS)
required_anomaly_values = [
    "affected_count", "severity", "owner", "status",
    "handling_decision", "likely_impact",
]
if anomaly_register.empty or anomaly_register[required_anomaly_values].isna().any().any():
    raise RuntimeError(
        "The non-blocking anomaly register is empty or missing required values."
    )

display(anomaly_register)
print("BLOCKING ANOMALIES: 0 unresolved")
print(f"NON-BLOCKING ANOMALIES PRESERVED: {len(anomaly_register)}")
print("ANOMALY TRIAGE: COMPLETE")


,check,passed,status
0,source identity,True,resolved
1,source schema,True,resolved
2,feature roles,True,resolved
3,frozen profile reproduction,True,resolved
4,identifier and dataset integrity,True,resolved
5,numeric ranges,True,resolved
6,distribution summaries,True,resolved
7,continuous target relationships,True,resolved
8,categorical target relationships,True,resolved


,id,description,affected_count,severity,owner,status,evidence,handling_decision,likely_impact
0,EDA-NB-001,Records containing at least one rare categoric...,14721,medium,feature engineering / modeling,non-blocking; handling defined,523 levels below 188 rows across 39 categorica...,Retain source values; fit any rare-level pooli...,Small groups can yield unstable target summari...
1,EDA-NB-002,Categorical fields with one level above 99% share,31,low,feature engineering / modeling,non-blocking; handling defined,cat15=0.999819; cat17=0.993049; cat18=0.994759...,Retain the fields for reproducibility; assess ...,Near-constant predictors may add little signal...
2,EDA-NB-003,Numeric fields with absolute skewness above 1,2,medium,EDA / modeling,non-blocking; handling defined,cont9: skewness=1.072429; loss: skewness=3.794958,Preserve raw values; use robust summaries and ...,"Means, Pearson correlations, and fitted object..."
3,EDA-NB-004,Row-field observations outside 1.5 x IQR fences,27547,medium,EDA / modeling,non-blocking; handling defined,cont7: 2659 (1.411973%); cont9: 13194 (7.00623...,"Treat as distribution-tail evidence, not error...",Tail observations may increase sensitivity of ...
4,EDA-NB-005,Predictors with quantile bins reduced by tied ...,2,low,EDA workflow,resolved,cont2: 9 of 10 bins; cont5: 8 of 10 bins,"Use qcut with duplicates='drop', retain the ac...",The affected predictors have fewer comparison ...
5,EDA-NB-006,Categorical fields lacking two supported level...,9,medium,EDA / modeling,non-blocking; interpretation constrained,cat15: 1 supported of 2 observed levels; cat22...,Retain all level summaries but do not make str...,Apparent target differences are driven by spar...


BLOCKING ANOMALIES: 0 unresolved
NON-BLOCKING ANOMALIES PRESERVED: 6
ANOMALY TRIAGE: COMPLETE


## Publish Checklist Evidence

Publish only the derived evidence needed to substantiate the requested checklist: the reproduced source profile, consolidated validation results, numeric ranges, categorical support, distribution summaries, target-relationship summaries, and anomaly register. No source rows are modified or exported.

Evidence publication is allowed only after every blocking check is resolved. Write the files through a temporary staging directory, replace the final files only after all writes succeed, then read them back and verify their schemas and row counts. Record the frozen source hash and artifact inventory in a run manifest so the evidence can be traced to this exact source.

In [16]:
if not blocking_resolution_results["passed"].all():
    raise RuntimeError("Evidence publication blocked by unresolved checks.")

def standardize_validation(
    category, frame, check_column, observed_column, expected_column,
    status_column="status",
):
    standardized = frame[
        [check_column, observed_column, expected_column, status_column]
    ].copy()
    standardized.columns = ["check", "observed", "expected", "status"]
    standardized.insert(0, "category", category)
    return standardized

coverage_result = pd.DataFrame(
    [
        {
            "category": "feature roles",
            "check": "one-to-one column coverage",
            "observed": coverage_passes,
            "expected": True,
            "status": "pass" if coverage_passes else "fail",
        }
    ]
)
blocking_validation = blocking_resolution_results.rename(
    columns={"passed": "observed"}
).copy()
blocking_validation.insert(0, "category", "blocking resolution")
blocking_validation["expected"] = True
blocking_validation["status"] = np.where(
    blocking_validation["observed"], "pass", "fail"
)
blocking_validation = blocking_validation[
    ["category", "check", "observed", "expected", "status"]
]

validation_results = pd.concat(
    [
        standardize_validation(
            "source identity", source_identity_results,
            "check", "observed", "expected",
        ),
        standardize_validation(
            "source schema", schema_results,
            "check", "observed", "expected",
        ),
        standardize_validation(
            "feature roles", role_results,
            "role", "observed_count", "expected_count",
        ),
        coverage_result,
        standardize_validation(
            "profile reproduction", profile_reproduction_results,
            "check", "observed", "expected",
        ),
        standardize_validation(
            "dataset integrity", integrity_results,
            "check", "observed", "expected",
        ),
        blocking_validation,
    ],
    ignore_index=True,
)

if not validation_results["status"].eq("pass").all():
    raise RuntimeError("Validation evidence contains a failed check.")

evidence_tables = {
    "source_profile.csv": reproduced_profile,
    "validation_results.csv": validation_results,
    "numeric_ranges.csv": numeric_range_results,
    "categorical_support.csv": categorical_support_results,
    "distribution_summary.csv": distribution_results,
    "continuous_target_relationships.csv": continuous_target_relationships,
    "continuous_quantile_bins.csv": continuous_quantile_bin_results,
    "categorical_target_relationships.csv": categorical_target_relationships,
    "categorical_level_target_relationships.csv": (
        categorical_level_target_results
    ),
    "anomaly_register.csv": anomaly_register,
}

EVIDENCE_DIR.parent.mkdir(parents=True, exist_ok=True)
staging_dir = Path(
    tempfile.mkdtemp(
        prefix=".data_quality_evidence_staging_",
        dir=EVIDENCE_DIR.parent,
    )
)

try:
    artifact_manifest = {}
    for filename, table in evidence_tables.items():
        staged_path = staging_dir / filename
        table.to_csv(staged_path, index=False)
        staged_table = pd.read_csv(staged_path)

        if len(staged_table) != len(table):
            raise RuntimeError(f"Staged row-count mismatch: {filename}")
        if staged_table.columns.tolist() != table.columns.tolist():
            raise RuntimeError(f"Staged schema mismatch: {filename}")

        artifact_manifest[filename] = {
            "rows": len(table),
            "columns": table.columns.tolist(),
            "sha256": calculate_sha256(staged_path),
        }

    run_manifest = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_path": str(DATA_PATH.relative_to(REPO_ROOT)),
        "source_bytes": observed_source_bytes,
        "source_sha256": observed_source_sha256,
        "source_rows": observed_rows,
        "source_columns": observed_columns,
        "python_version": platform.python_version(),
        "pandas_version": pd.__version__,
        "artifacts": artifact_manifest,
    }
    staged_manifest_path = staging_dir / "run_manifest.json"
    with staged_manifest_path.open("w", encoding="utf-8") as manifest_file:
        json.dump(run_manifest, manifest_file, indent=2)

    with staged_manifest_path.open("r", encoding="utf-8") as manifest_file:
        staged_manifest = json.load(manifest_file)
    if staged_manifest["source_sha256"] != EXPECTED_SOURCE_SHA256:
        raise RuntimeError("Staged manifest source hash mismatch.")
    if set(staged_manifest["artifacts"]) != set(evidence_tables):
        raise RuntimeError("Staged manifest artifact inventory mismatch.")

    EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
    for staged_path in staging_dir.iterdir():
        os.replace(staged_path, EVIDENCE_DIR / staged_path.name)
finally:
    shutil.rmtree(staging_dir, ignore_errors=True)

publication_rows = []
for filename, table in evidence_tables.items():
    published_path = EVIDENCE_DIR / filename
    published_table = pd.read_csv(published_path)
    row_count_matches = len(published_table) == len(table)
    schema_matches = published_table.columns.tolist() == table.columns.tolist()
    hash_matches = (
        calculate_sha256(published_path)
        == run_manifest["artifacts"][filename]["sha256"]
    )
    publication_rows.append(
        {
            "artifact": filename,
            "expected_rows": len(table),
            "observed_rows": len(published_table),
            "schema_matches": schema_matches,
            "hash_matches": hash_matches,
            "status": (
                "pass"
                if row_count_matches and schema_matches and hash_matches
                else "fail"
            ),
        }
    )

with (EVIDENCE_DIR / "run_manifest.json").open(
    "r", encoding="utf-8"
) as manifest_file:
    published_manifest = json.load(manifest_file)
manifest_matches = published_manifest == run_manifest

publication_results = pd.DataFrame(publication_rows)
display(publication_results)

if not publication_results["status"].eq("pass").all() or not manifest_matches:
    raise RuntimeError("Published evidence failed final read-back validation.")

print(f"EVIDENCE DIRECTORY: {EVIDENCE_DIR}")
print(f"CSV ARTIFACTS PUBLISHED: {len(evidence_tables)}")
print("RUN MANIFEST: PASS")
print("EVIDENCE PUBLICATION: COMPLETE")


,artifact,expected_rows,observed_rows,schema_matches,hash_matches,status
0,source_profile.csv,132,132,True,True,pass
1,validation_results.csv,28,28,True,True,pass
2,numeric_ranges.csv,16,16,True,True,pass
3,categorical_support.csv,116,116,True,True,pass
4,distribution_summary.csv,15,15,True,True,pass
5,continuous_target_relationships.csv,14,14,True,True,pass
6,continuous_quantile_bins.csv,137,137,True,True,pass
7,categorical_target_relationships.csv,116,116,True,True,pass
8,categorical_level_target_relationships.csv,1139,1139,True,True,pass
9,anomaly_register.csv,6,6,True,True,pass


EVIDENCE DIRECTORY: /Users/ragibn/Library/CloudStorage/OneDrive-Personal/Documents/Programs/BTT - AI Studio Fall/Allstate-1A-predicting-auto-claims-severity/notebooks/final-deliverables/September/Gate 3/data_quality_evidence
CSV ARTIFACTS PUBLISHED: 10
RUN MANIFEST: PASS
EVIDENCE PUBLICATION: COMPLETE
